# AUV v0.3.1 — hybryda C

Wersja po krytyce v0.3.0. Trzy zmiany koncepcyjne:

1. **Dwa komponenty zamiast jednego**:
   - *Real* — ważona suma realnych wartości zasobów (food + energy + metals)
   - *Fiat* — konsensus CPI głównych walut (USA, EUR, JP, UK, CH)
   - AUV = real / fiat × 100

2. **Lepsze wagi**: 30% food + 40% energy + 30% metals (zamiast 50/50 z głowy).

3. **Bezwzględne wartości walut**: pokazujemy *ile naprawdę* USD/EUR/PLN trzeba na 1 AUV w czasie, bez normalizacji każdej waluty do 100 w t_0. To było główne źle zaprojektowanie poprzedniego notebooka.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)

PROTOTYP_ROOT = Path.cwd().parent
if str(PROTOTYP_ROOT) not in sys.path:
    sys.path.insert(0, str(PROTOTYP_ROOT))

from src.auv_v3_hybrid import construct_auv_v3_hybrid

DATA_DIR = PROTOTYP_ROOT / 'data' / 'processed'
FIG_DIR = PROTOTYP_ROOT / 'outputs' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 9

monthly = pd.read_parquet(DATA_DIR / 'monthly.parquet')
print(f'Loaded monthly: {monthly.shape}')

# Sprawdź, czy CPI są pobrane
cpi_cols = [c for c in monthly.columns if c.startswith('CPI_')]
print(f'Dostępne CPI: {cpi_cols}')

## 1. Komponent fiat — konsensus CPI G5

Średnia ważona PKB pięciu największych walut. To pokazuje, *jak fiat-jako-całość się rozcieńcza* — niezależnie metodologicznie od naszych obliczeń surowcowych.

In [ ]:
result = construct_auv_v3_hybrid(monthly, t0='2017-06-30')

fig, ax = plt.subplots(figsize=(14, 6))
fiat = result.fiat_component.dropna()
ax.plot(fiat.index, fiat.values, color='#aa3333', linewidth=2, label='Konsensus CPI G5')

# Pokażmy też pojedyncze CPI dla porównania
for cpi_name, color in [('CPI_USA', '#3366aa'), ('CPI_EU', '#66aa66'),
                         ('CPI_JP', '#aa6633'), ('CPI_CH', '#333333')]:
    if cpi_name in monthly.columns:
        s = monthly[cpi_name].dropna()
        if not s.empty:
            s_norm = s / s.iloc[0] * 100
            ax.plot(s_norm.index, s_norm.values, alpha=0.5, linewidth=1, label=cpi_name)

ax.set_title('Komponent fiat — konsensus CPI G5\n(jak średnio rośnie cena koszyka konsumenckiego w głównych walutach)')
ax.set_ylabel('Wartość (1996-01 = 100)')
ax.axhline(100, color='gray', alpha=0.3)
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / '15_hybrid_fiat.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nKonsensus CPI G5:')
print(f'  1996-01: {fiat.iloc[0]:.1f}')
print(f'  2025-12: {fiat.iloc[-1]:.1f}')
print(f'  Wzrost: {(fiat.iloc[-1]/fiat.iloc[0]-1)*100:+.1f}% (czyli realna deprecjacja koszyka G5 w 30 lat)')

## 2. Komponent real — ważona suma realnych wartości

Food (30%) + Energy (40%) + Metals (30%), gdzie metals to średnia ceny miedzi/aluminium/żelaza/niklu/cynku/ołowiu (bez produkcji, bo nie mamy USGS jeszcze — przyjmujemy, że produkcja metali rośnie liniowo z populacją).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
real = result.real_component.dropna()
ax.plot(real.index, real.values, color='#225588', linewidth=2, label='Komponent realny')
ax.axhline(100, color='gray', linestyle='--', alpha=0.5, label='t_0 = 100')
ax.set_title('Komponent realny — ważona suma realnych wartości zasobów\n(30% food + 40% energy + 30% metals)')
ax.set_ylabel('Wartość (t_0 = 100)')
ax.legend()
ax.grid(alpha=0.3)

for date, label in [('2008-09-01', 'Lehman'), ('2014-06-01', 'Oil collapse'),
                     ('2020-03-01', 'COVID'), ('2022-02-01', 'Ukraine')]:
    d = pd.Timestamp(date)
    if d in real.index:
        ax.annotate(label, xy=(d, real.loc[d]), xytext=(5, 15),
                    textcoords='offset points', fontsize=8,
                    arrowprops=dict(arrowstyle='->', alpha=0.4))

plt.tight_layout()
plt.savefig(FIG_DIR / '16_hybrid_real.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. AUV = real / fiat — finalny wskaźnik

**Interpretacja:**
- AUV > 100: realne zasoby drożeją szybciej niż fiat się rozcieńcza (kryzysy podażowe)
- AUV < 100: świat realnie się bogaci (postęp wyprzedza inflację)
- AUV ≈ 100: dwa procesy się równoważą

In [ ]:
auv = result.auv.dropna()

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(real.index, real.values, label='Real (zasoby)', color='#225588', linewidth=1.3, alpha=0.7)
ax.plot(fiat.index, fiat.values, label='Fiat (CPI G5)', color='#aa3333', linewidth=1.3, alpha=0.7)
ax.plot(auv.index, auv.values, label='AUV = Real / Fiat × 100', color='#222244', linewidth=2.5)

ax.axhline(100, color='gray', linestyle='--', alpha=0.4)
ax.axvline(pd.Timestamp('2017-06-30'), color='red', linestyle=':', alpha=0.4)
ax.set_title('AUV v0.3.1 (hybryda C) — wszystkie komponenty razem')
ax.set_ylabel('Wartość (t_0 = 100)')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / '17_hybrid_auv_combined.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nKluczowe punkty AUV v0.3.1:')
for date in ['2000-01-31', '2008-09-30', '2014-12-31', '2017-06-30',
             '2020-03-31', '2022-02-28', '2025-12-31']:
    try:
        v = auv.loc[date]
        r = real.loc[date]
        f = fiat.loc[date]
        print(f'  {date}: AUV={v:6.1f}  real={r:6.1f}  fiat={f:6.1f}  → {"realne droższe" if v>100 else "realne tańsze"}')
    except KeyError:
        pass

## 4. Bezwzględna wartość 1 AUV w różnych walutach

**Poprawka z poprzedniej iteracji**: pokazujemy *prawdziwe* liczby, bez normalizacji każdej waluty do 100 w t_0. To ujawnia, jak naprawdę PLN, EUR, JPY zachowały się wobec AUV — i pokazuje wyraźne różnice między walutami silnymi i słabymi.

In [ ]:
# Wartość 1 AUV w USD: definiujemy tak, że w t_0 = 1 USD = 0.01 AUV (czyli 1 AUV = 100 USD).
# Wtedy AUV/USD(t) = AUV(t) — bo z definicji AUV jest indeksem w odniesieniu do USD.

# Dla pozostałych walut: 1 AUV w walucie X = AUV(t) × (jednostek X za 1 USD)(t)
# Czyli mnożymy AUV przez kurs waluty

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

currencies_info = [
    ('USD', None, '#222244', 'USD'),
    ('EUR', 'EUR', '#33aa66', 'EUR'),
    ('PLN', 'PLN', '#cc3366', 'PLN'),
    ('JPY', 'JPY', '#aa6633', 'JPY'),
]

for ax_idx, (label, col, color, currency_name) in enumerate(currencies_info):
    ax = axes[ax_idx // 2, ax_idx % 2]
    if col is None:
        # USD jest jednostką bazową AUV — wartość 1 AUV w USD to po prostu AUV(t)
        values = auv
    else:
        if col not in monthly.columns:
            ax.text(0.5, 0.5, f'Brak danych {col}', ha='center', va='center')
            continue
        rate = monthly[col].dropna()
        common = auv.index.intersection(rate.index)
        values = auv.loc[common] * rate.loc[common]

    ax.plot(values.index, values.values, color=color, linewidth=2)
    ax.set_title(f'Ile {currency_name} za 1 AUV w czasie\n'
                 f'1996: {values.dropna().iloc[0]:.2f}  →  2025: {values.dropna().iloc[-1]:.2f}  '
                 f'({(values.dropna().iloc[-1]/values.dropna().iloc[0]-1)*100:+.0f}%)')
    ax.set_ylabel(f'{currency_name} za 1 AUV')
    ax.grid(alpha=0.3)
    ax.axvline(pd.Timestamp('2017-06-30'), color='red', linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig(FIG_DIR / '18_hybrid_currencies_absolute.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Porównanie krajów — realna deprecjacja walut

Tabelka pokazująca, ile realnie straciła każda waluta od 1996 do 2025 wobec AUV. Tym razem widać prawdziwe różnice — PLN, BRL, TRY tracą drastycznie, CHF i JPY trzymają się dobrze.

In [ ]:
# Tabela realnej deprecjacji (1996 → 2025)
currencies_to_show = ['USD', 'EUR', 'JPY', 'GBP', 'CHF', 'CNY', 'INR', 'BRL',
                       'MXN', 'KRW', 'AUD', 'CAD', 'SGD', 'ZAR', 'SEK', 'PLN', 'CZK', 'HUF']

rows = []
for currency in currencies_to_show:
    if currency == 'USD':
        try:
            v0 = auv.dropna().iloc[0]
            v1 = auv.dropna().iloc[-1]
            rows.append({'waluta': 'USD', '1996': v0, '2025': v1,
                         'zmiana%': (v1/v0 - 1) * 100})
        except IndexError:
            pass
    elif currency in monthly.columns:
        rate = monthly[currency].dropna()
        common = auv.index.intersection(rate.index)
        if len(common) < 12:
            continue
        v = (auv.loc[common] * rate.loc[common]).dropna()
        if v.empty:
            continue
        v0 = v.iloc[0]
        v1 = v.iloc[-1]
        rows.append({'waluta': currency, '1996': v0, '2025': v1,
                     'zmiana%': (v1/v0 - 1) * 100})

tab = pd.DataFrame(rows).sort_values('zmiana%', ascending=False)
print('Realna deprecjacja walut wobec AUV v0.3.1 (cały okres dostępnych danych):\n')
for _, r in tab.iterrows():
    print(f"  {r['waluta']:6s}  {r['1996']:>12,.2f} →  {r['2025']:>14,.2f}  ({r['zmiana%']:+8.1f}%)")

# Wykres słupkowy
fig, ax = plt.subplots(figsize=(12, 7))
tab_plot = tab.set_index('waluta')['zmiana%']
colors = ['#cc3333' if v > 100 else '#3399cc' if v < 50 else '#cccccc' for v in tab_plot.values]
tab_plot.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Realna deprecjacja walut wobec AUV v0.3.1 — cały okres (1996/2002 → 2025)')
ax.set_xlabel('Zmiana % (im więcej, tym waluta straciła więcej)')
ax.axvline(0, color='black', alpha=0.3)
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(FIG_DIR / '19_hybrid_depreciation_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

## Wnioski

Najważniejsze pytania, które warto omówić po obejrzeniu wykresów:

1. **Czy hybryda C daje stabilniejszy AUV niż v0.3.0?** (wykres 17 — wszystko razem)
2. **Czy AUV ma sensowną interpretację: <100 to dobry znak (postęp wyprzedza inflację), >100 to zły (szok podażowy)?** (wykres 17)
3. **Czy waluty wreszcie pokazują znaczące różnice?** (wykres 18, 19 — tabela rankingu)
4. **Czy konsensus CPI G5 zachowuje się tak, jak intuicja podpowiada?** (wykres 15)
5. **Czy komponent realny pokazuje rozsądne reakcje na kryzysy?** (wykres 16)